## Feature Engineering to prep data for modeling

##### Import Necessary Packages

In [4]:
import pandas as pd
import os
import time
import requests

##### Import MovieLens 32M ratings data

In [47]:
ml_ratings = pd.read_csv("../data/movielens/ratings.csv")
ml_ratings.head()
#around 160 reviews per user (I have ~250)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


##### Import ML32M Links dataset to get TMDBID for TMDB data enrichment later on

In [48]:
ml_links = pd.read_csv("../data/movielens/links.csv")
ml_links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


##### Import ML32M Movie data

In [42]:
ml_movies = pd.read_csv("../data/movielens/movies_with_tmdbId.csv")
ml_movies.head()

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0


In [7]:
ml_movies['movieId'].nunique()
# There are 87,585 movies in the dataset

87585

#### Enrich MovieLens data

##### This will give us key metadata from the movie's genre, cast, description, poster, etc. for our content based model

In [ ]:
from dotenv import load_dotenv

load_dotenv("../.env")

TMDB_TOKEN = os.getenv("TMDB_TOKEN")

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

In [ ]:
def get_tmdb_features(tmdb_id):

    if pd.isna(tmdb_id):
        return None

    tmdb_id = int(tmdb_id)

    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

    params = {
        "language": "en-US",
        "append_to_response": "credits,keywords"
    }

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            params=params,
            timeout=15
        )

        response.raise_for_status()

        movie = response.json()

        # -----------------------------
        # Genres
        # -----------------------------

        genres = [
            genre["name"]
            for genre in movie.get("genres", [])
        ]


        # -----------------------------
        # Director
        # -----------------------------

        crew = movie.get(
            "credits", {}
        ).get("crew", [])

        directors = [
            person["name"]
            for person in crew
            if person.get("job") == "Director"
        ]


        # -----------------------------
        # Top cast
        # -----------------------------

        cast = movie.get(
            "credits", {}
        ).get("cast", [])

        top_cast = [
            person["name"]
            for person in cast[:5]
        ]


        # -----------------------------
        # Keywords
        # -----------------------------

        keywords = [
            keyword["name"]
            for keyword in movie.get(
                "keywords", {}
            ).get("keywords", [])
        ]


        # -----------------------------
        # Poster
        # -----------------------------

        poster_path = movie.get("poster_path")

        poster_url = (
            f"https://image.tmdb.org/t/p/w500{poster_path}"
            if poster_path
            else None
        )


        return {
            "tmdb_overview": movie.get("overview"),
            "tmdb_genres": "|".join(genres),
            "director": "|".join(directors),
            "cast": "|".join(top_cast),
            "keywords": "|".join(keywords),
            "release_date": movie.get("release_date"),
            "runtime": movie.get("runtime"),
            "popularity": movie.get("popularity"),
            "vote_average": movie.get("vote_average"),
            "vote_count": movie.get("vote_count"),
            "poster_url": poster_url
        }

    except requests.RequestException as e:

        print(
            f"TMDB error for {tmdb_id}: {e}"
        )

        return None

In [ ]:
movie_features = []

total = len(ml_movies)

for i, tmdb_id in enumerate(
    ml_movies["tmdbId"]
):

    features = get_tmdb_features(
        tmdb_id
    )

    movie_features.append(
        features or {}
    )

    if (i + 1) % 100 == 0:
        print(
            f"{i + 1:,} / {total:,} movies enriched"
        )

    # Small delay to be polite to API
    time.sleep(0.03)

In [ ]:
tmdb_features = pd.DataFrame(
    movie_features
)

movies_enriched = pd.concat(
    [
        ml_movies.reset_index(drop=True),
        tmdb_features
    ],
    axis=1
)

movies_enriched.head()

##### Save enriched movielens data since it takes a long time to run the API call for over 85,000 movies

In [ ]:
tmdb_features = pd.DataFrame(
    movie_features
)

##### Read in and look at enriched movie lens data

In [43]:
movies_enriched = pd.read_parquet("../data/movie_with_features.parquet")

In [44]:
movies_enriched.head()

,movieId,title,genres,imdbId,tmdbId,tmdb_overview,tmdb_genres,director,cast,keywords,release_date,runtime,popularity,vote_average,vote_count,poster_url
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"Led by Woody, Andy's toys live happily in his ...",Family|Comedy|Animation|Adventure,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,rescue|friendship|mission|jealousy|villain|bul...,1995-11-22,81.0,41.2575,7.989,20565.0,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0,When siblings Judy and Peter discover an encha...,Adventure|Fantasy|Family,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,based on novel or book|giant insect|board game...,1995-12-15,104.0,4.2650,7.249,11586.0,https://image.tmdb.org/t/p/w500/bdHG5Mo83VPobe...
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,A family wedding reignites the ancient feud be...,Romance|Comedy,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,fishing|sequel|old man|best friend|wedding|ita...,1995-12-22,101.0,5.3009,6.466,444.0,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama|Romance,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,based on novel or book|single mother|divorce|a...,1995-12-22,127.0,4.6529,6.200,213.0,https://image.tmdb.org/t/p/w500/4wjGMwPsdlvi02...
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,Just when George Banks has recovered from his ...,Comedy|Family,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,daughter|baby|parent child relationship|midlif...,1995-12-08,106.0,6.4058,6.272,841.0,https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...


##### How recent is this data? What is the latest release date?

In [45]:
movies_enriched["release_date"].max()

'2026-07-25'

##### Add TMDB ID to the movielens movies

In [49]:
ml_ratings = pd.merge(ml_ratings, ml_links[["movieId", "tmdbId"]], how="left", on="movieId")

##### Add ratings to movielens movies

In [12]:
ml32M_with_ratings = pd.merge(ml_ratings, ml_movies, on="tmdbId", how="left")

In [ ]:
ml32M_with_ratings.drop(columns=["movieId_y"], inplace=True) # removes extra column created in the join

In [15]:
ml32M_with_ratings.rename(columns={"movieId_x": "movieId"}, inplace=True) # renames the movieId column to the original name

In [17]:
ml32M_with_ratings.shape

(32594948, 8)

In [50]:
ml32M_with_ratings = pd.read_parquet("../data/movielens_ratings_w_tmdbId.parquet")

In [51]:
ml32M_with_ratings.head()

,userId,tmdbId,rating
0,1,4584.0,4.0
1,1,451.0,1.0
2,1,902.0,2.0
3,1,37557.0,5.0
4,1,63.0,5.0


### Now we have a dataset of 32M movie ratings with each row having the user's unique ID, the movie's unique TMDB ID so that we can get enriched movie data, and the user's rating of that movie. We also have a catalog of all the 87K+ movies enriched with TMDB data

### Create Processed Text Field for TF-IDF

#### Join ml32M_rankings with enriched movie data on tmdbid to get rankings and metadata in one table

In [52]:
movies_enriched.head()

,movieId,title,genres,imdbId,tmdbId,tmdb_overview,tmdb_genres,director,cast,keywords,release_date,runtime,popularity,vote_average,vote_count,poster_url
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"Led by Woody, Andy's toys live happily in his ...",Family|Comedy|Animation|Adventure,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,rescue|friendship|mission|jealousy|villain|bul...,1995-11-22,81.0,41.2575,7.989,20565.0,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0,When siblings Judy and Peter discover an encha...,Adventure|Fantasy|Family,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,based on novel or book|giant insect|board game...,1995-12-15,104.0,4.2650,7.249,11586.0,https://image.tmdb.org/t/p/w500/bdHG5Mo83VPobe...
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,A family wedding reignites the ancient feud be...,Romance|Comedy,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,fishing|sequel|old man|best friend|wedding|ita...,1995-12-22,101.0,5.3009,6.466,444.0,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama|Romance,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,based on novel or book|single mother|divorce|a...,1995-12-22,127.0,4.6529,6.200,213.0,https://image.tmdb.org/t/p/w500/4wjGMwPsdlvi02...
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,Just when George Banks has recovered from his ...,Comedy|Family,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,daughter|baby|parent child relationship|midlif...,1995-12-08,106.0,6.4058,6.272,841.0,https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...


#### Define functions to parse movie metadata to get features for genre, cast, keywords, director

In [53]:
def process_genres(genres):
    if pd.isna(genres):
        return ""

    return " ".join(
        [
            f"genre_{g.strip().lower()}"
            for g in genres.split("|")
        ]
    )

In [54]:
def process_keywords(keywords):
    if pd.isna(keywords):
        return ""

    return " ".join(
        [
            f"keyword_{k.strip().lower().replace(' ', '_')}"
            for k in keywords.split("|")
        ]
    )

In [55]:
def process_director(director):
    if pd.isna(director):
        return ""

    return " ".join(
        [
            f"director_{d.strip().lower().replace(' ', '_')}"
            for d in director.split("|")
        ]
    )

In [56]:
def process_cast(cast):
    if pd.isna(cast):
        return ""

    return " ".join(
        [
            f"actor_{c.strip().lower().replace(' ', '_')}"
            for c in cast.split("|")
        ]
    )

In [57]:
movies_enriched["content_text"] = (
    movies_enriched["tmdb_genres"].apply(process_genres)
    + " "
    + movies_enriched["keywords"].apply(process_keywords)
    + " "
    + movies_enriched["director"].apply(process_director)
    + " "
    + movies_enriched["cast"].apply(process_cast)
    + " "
    + movies_enriched["tmdb_overview"].fillna("").str.lower()
)

In [58]:
movies_enriched.head()

,movieId,title,genres,imdbId,tmdbId,tmdb_overview,tmdb_genres,director,cast,keywords,release_date,runtime,popularity,vote_average,vote_count,poster_url,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,"Led by Woody, Andy's toys live happily in his ...",Family|Comedy|Animation|Adventure,John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,rescue|friendship|mission|jealousy|villain|bul...,1995-11-22,81.0,41.2575,7.989,20565.0,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...,genre_family genre_comedy genre_animation genr...
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0,When siblings Judy and Peter discover an encha...,Adventure|Fantasy|Family,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,based on novel or book|giant insect|board game...,1995-12-15,104.0,4.2650,7.249,11586.0,https://image.tmdb.org/t/p/w500/bdHG5Mo83VPobe...,genre_adventure genre_fantasy genre_family key...
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,A family wedding reignites the ancient feud be...,Romance|Comedy,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,fishing|sequel|old man|best friend|wedding|ita...,1995-12-22,101.0,5.3009,6.466,444.0,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...,genre_romance genre_comedy keyword_fishing key...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama|Romance,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,based on novel or book|single mother|divorce|a...,1995-12-22,127.0,4.6529,6.200,213.0,https://image.tmdb.org/t/p/w500/4wjGMwPsdlvi02...,genre_comedy genre_drama genre_romance keyword...
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,Just when George Banks has recovered from his ...,Comedy|Family,Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,daughter|baby|parent child relationship|midlif...,1995-12-08,106.0,6.4058,6.272,841.0,https://image.tmdb.org/t/p/w500/rj4LBtwQ0uGrpB...,genre_comedy genre_family keyword_daughter key...


In [59]:
# drop duplicate movies
movies_enriched = movies_enriched.drop_duplicates(
    subset="tmdbId"
)

In [60]:
# check for null content fields
movies_enriched["content_text"].isna().sum()

np.int64(0)

In [61]:
movies_enriched.to_parquet("../data/content_modeling_data.parquet", index=False)